<a href="https://colab.research.google.com/github/trisha123789/RAG_WITH_PINECODE_VECTORDATABASE/blob/main/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q \
pinecone \
langchain \
langchain-community \
langchain-huggingface \
sentence-transformers \
pypdf

In [2]:
!pip install -U langchain langchain-text-splitters

In [4]:
!pip install -U langchain langchain-core langchain-text-splitters langchain-huggingface langchain-pinecone pypdf sentence-transformers pinecone

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.4/557.4 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.4/596.4 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.6/587.6 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.4/120.4 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.3/259.3 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 4.6 MB/s eta 0:00:00
  Attempting uninstall: packaging
    Found existing installation: packaging 26.2
    Uninstalling packaging-26.2:
      Successfully uninstalled packaging-26.2
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: pinecone
    Found existing installation: pinecone 9.1.0
    Uninstalling pinecone-9.1.0:
      Successful

In [1]:
from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings

from pinecone import Pinecone, ServerlessSpec

from langchain_pinecone import PineconeVectorStore

import os

/tmp/ipykernel_5549/1659727124.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
loader = PyPDFLoader("RAGPaper.pdf")

documents = loader.load()

In [3]:
print(len(documents))

19


In [4]:
print(documents[0].page_content)

Retrieval-Augmented Generation for
Knowledge-Intensive NLP Tasks
Patrick Lewis†‡, Ethan Perez⋆,
Aleksandra Piktus†, Fabio Petroni†, Vladimir Karpukhin†, Naman Goyal†, Heinrich Küttler†,
Mike Lewis†, Wen-tau Yih†, Tim Rocktäschel†‡, Sebastian Riedel†‡, Douwe Kiela†
†Facebook AI Research;‡University College London;⋆New York University;
plewis@fb.com
Abstract
Large pre-trained language models have been shown to store factual knowledge
in their parameters, and achieve state-of-the-art results when ﬁne-tuned on down-
stream NLP tasks. However, their ability to access and precisely manipulate knowl-
edge is still limited, and hence on knowledge-intensive tasks, their performance
lags behind task-speciﬁc architectures. Additionally, providing provenance for their
decisions and updating their world knowledge remain open research problems. Pre-
trained models with a differentiable access mechanism to explicit non-parametric
memory have so far been only investigated for extractive downstream tas

In [5]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

docs = splitter.split_documents(documents)

In [6]:
len(docs)

183

In [7]:
print(docs[0].page_content)

Retrieval-Augmented Generation for
Knowledge-Intensive NLP Tasks
Patrick Lewis†‡, Ethan Perez⋆,
Aleksandra Piktus†, Fabio Petroni†, Vladimir Karpukhin†, Naman Goyal†, Heinrich Küttler†,
Mike Lewis†, Wen-tau Yih†, Tim Rocktäschel†‡, Sebastian Riedel†‡, Douwe Kiela†
†Facebook AI Research;‡University College London;⋆New York University;
plewis@fb.com
Abstract
Large pre-trained language models have been shown to store factual knowledge


In [8]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [9]:
vector = embedding_model.embed_query("What is RAG?")

print(len(vector))

384


In [10]:
PINECONE_API_KEY = "your-api-key"

pc = Pinecone(api_key=PINECONE_API_KEY)

In [11]:
index_name = "rag-paper"

In [12]:
if index_name not in pc.list_indexes().names():

    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

In [13]:
vectorstore = PineconeVectorStore(
    index_name=index_name,
    embedding=embedding_model,
    pinecone_api_key=PINECONE_API_KEY
)

In [14]:
vectorstore.add_documents(docs)

['d7d17033-2d6c-48e6-a439-61936f1deca6',
 'bd4c5a86-df49-48e4-9a40-2a5af0f1be6a',
 'a71e6d55-4251-4d55-8737-fefb813fbd03',
 '413c50e2-6056-4a4f-a353-877e52638416',
 '35a57aa3-8bbf-4097-a331-39c5f50db313',
 '0d8f77a0-94be-42eb-bab9-5a41e2fe55d6',
 '40014134-a0a5-4a03-9209-5e9db2192157',
 'd72bae51-b87a-4209-bd6b-2692b4b2afe5',
 'cc729623-9984-46d4-bf9a-be084d65debb',
 'c8aafbe3-3657-4f4e-91c4-3abc7867990e',
 '9906a4a2-196c-45d1-b990-368be140ddf1',
 'b81286bb-93eb-4b63-9948-9a1b3c3bc686',
 'afba2f32-2a8f-4ead-8df1-7ee3243333c4',
 '2dad24f5-421f-4dd8-9245-4acade938ee1',
 '58f7c741-a567-4e36-8cb9-eb5ab618cb97',
 '05484be6-2961-4e4c-917a-65c56b704113',
 '4bf76d8b-b7b4-494d-b816-aa2699a760d0',
 '4486a93b-1cd3-4490-8777-ecf68edc6b22',
 '7c396674-7a83-485b-b260-69600c87e1bf',
 'd0ced381-1b1b-4ebf-8885-8632382f52a8',
 'd73ce0d0-fb70-4873-9f27-167a41488eee',
 '08364d9e-ad88-4e2d-b65a-1677c2c2666e',
 '7c782cf8-19e9-4ba0-bbad-f8a335ccff54',
 '7a08d293-f26a-4822-b7cd-cad9a6d90086',
 'f1540a4c-fa2e-

In [15]:
query = "What is Retrieval Augmented Generation?"

results = vectorstore.similarity_search(query, k=3)

In [16]:
for i, doc in enumerate(results):
    print("="*60)
    print(i+1)
    print(doc.page_content)

1
In this work, we presented hybrid generation models with access to parametric and non-parametric
memory. We showed that our RAG models obtain state of the art results on open-domain QA. We
found that people prefer RAG’s generation over purely parametric BART, ﬁnding RAG more factual
and speciﬁc. We conducted an thorough investigation of the learned retrieval component, validating
its effectiveness, and we illustrated how the retrieval index can be hot-swapped to update the model
2
lags behind task-speciﬁc architectures. Additionally, providing provenance for their
decisions and updating their world knowledge remain open research problems. Pre-
trained models with a differentiable access mechanism to explicit non-parametric
memory have so far been only investigated for extractive downstream tasks. We
explore a general-purpose ﬁne-tuning recipe for retrieval-augmented generation
(RAG) — models which combine pre-trained parametric and non-parametric mem-
3
architecture, by learning a re

In [17]:
results = vectorstore.similarity_search_with_score(
    query,
    k=5
)

for doc, score in results:

    print(score)
    print(doc.page_content[:300])

0.598613739
In this work, we presented hybrid generation models with access to parametric and non-parametric
memory. We showed that our RAG models obtain state of the art results on open-domain QA. We
found that people prefer RAG’s generation over purely parametric BART, ﬁnding RAG more factual
and speciﬁc. We 
0.584068298
lags behind task-speciﬁc architectures. Additionally, providing provenance for their
decisions and updating their world knowledge remain open research problems. Pre-
trained models with a differentiable access mechanism to explicit non-parametric
memory have so far been only investigated for extract
0.552248955
architecture, by learning a retrieval module to augment pre-trained, generative language models.
Learned Retrieval There is signiﬁcant work on learning to retrieve documents in information
retrieval, more recently with pre-trained, neural language models [ 44, 26] similar to ours. Some
work optimize
0.535970211
In preliminary experiments, we observed that for 

In [18]:
while True:

    question = input("Ask : ")

    if question.lower() == "exit":
        break

    docs = vectorstore.similarity_search(question, k=3)

    print()

    for doc in docs:
        print(doc.page_content)
        print("-"*50)

Ask : what is rag?

paper, as well as HuggingFace for their help in open-sourcing code to run RAG models. The authors
would also like to thank Kyunghyun Cho and Sewon Min for productive discussions and advice. EP
thanks supports from the NSF Graduate Research Fellowship. PL is supported by the FAIR PhD
program.
References
[1] Payal Bajaj, Daniel Campos, Nick Craswell, Li Deng, Jianfeng Gao, Xiaodong Liu, Rangan
Majumder, Andrew McNamara, Bhaskar Mitra, Tri Nguyen, Mir Rosenberg, Xia Song, Alina
--------------------------------------------------
source, will probably never be entirely factual and completely devoid of bias. Since RAG can be
employed as a language model, similar concerns as for GPT-2 [50] are valid here, although arguably
to a lesser extent, including that it might be used to generate abuse, faked or misleading content in
the news or on social media; to impersonate others; or to automate the production of spam/phishing
content [54]. Advanced language models may also lead 

In [20]:
!pip install -q -U langchain-google-genai google-generativeai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.5/70.5 kB 4.5 MB/s eta 0:00:00


In [21]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [26]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key="your api key"
)

In [27]:
question = "Explain RAG."

docs = vectorstore.similarity_search(question, k=3)

context = "\n\n".join([d.page_content for d in docs])

In [28]:
prompt = f"""
Answer only using the context.

Context:

{context}

Question:

{question}
"""

In [29]:
response = llm.invoke(prompt)

print(response.content)

RAG can be employed as a language model.
